### Dataset and Task Metadata

In [8]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="hazelnut_spread_contaminant_detection",
    dataset_year="2020",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/45538",
    download_description="""
mkdir -p local-data-warehouse/hazelnut_spread_contaminant_detection/ && wget https://api.openml.org/data/download/22116506/dataset -O ./local-data-warehouse/hazelnut_spread_contaminant_detection/hazelnut.arff
""",
    # References
    academic_reference_bibtex=r"""@article{ricci2021machine,
  title={Machine-learning-based microwave sensing: A case study for the food industry},
  author={Ricci, Marco and {\v{S}}titi{\'c}, Bernardita and Urbinati, Luca and Di Guglielmo, Giuseppe and Vasquez, Jorge A Tob{\'o}n and Carloni, Luca P and Vipiana, Francesca and Casu, Mario R},
  journal={IEEE Journal on Emerging and Selected Topics in Circuits and Systems},
  volume={11},
  number={3},
  pages={503--514},
  year={2021},
  publisher={IEEE}
}
""",
    academic_reference_bibtex_key="ricci2021machine",
    license="CC BY-SA",
    data_tags=["IID"],
    curation_comments="""
- We select a features from a single frequency (10 GHz) as the authors also only considered this frequency for the final experiments.
- Anomaly: we use the publicly available dataset state, which is without preprocessing. Moreover, we were not able to inverse the original ordinal encoding of the label.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Contaminated",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Contaminated",
)

## Preprocessing

In [9]:
import pandas as pd
import arff

with open(f"{dataset_mold.path}/hazelnut.arff") as f:
    data = arff.load(f)
df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

target_feature = "Contaminated"
df = df.rename(columns={"class": target_feature})
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"}).astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [10]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,400
Columns: 31
Use sampling: False (sample size: 2,400)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['s21', 's15', 's14', 's23', 's16', 's32', 's31', 's34', 's12', 's24']
Rows remaining as candidates after top-10 filter: 0 (of 2,400)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [11]:
# Sample Rows
df_head

,s12,s13,s14,s15,s16,s21,s23,s24,s25,s26,s31,s32,s34,s35,s36,s41,s42,s43,s45,s46,s51,s52,s53,s54,s56,s61,s62,s63,s64,s65,Contaminated
0,-0.002261,-0.000834,-0.002984,0.000956,0.000467,0.000332,-0.000999,-0.000153,-0.000274,0.000140,-0.002366,-0.004813,0.000125,-0.000567,0.002555,0.000116,-0.000105,-0.000187,0.001833,0.002041,-0.001183,-0.000343,0.002354,-0.001799,-0.002190,-0.000968,0.000474,0.001126,0.001613,-0.002201,Yes
1,-0.002118,-0.000653,-0.002987,0.000289,-0.000962,-0.000529,-0.000912,-0.000180,-0.000749,-0.000288,-0.002417,-0.003836,-0.001068,-0.000814,0.002754,-0.000367,0.000042,0.000893,0.002286,0.001907,-0.001364,-0.000898,0.001614,-0.001789,-0.002132,-0.000739,0.000786,0.000843,0.002330,-0.001714,Yes
2,-0.002431,-0.000415,-0.004407,0.001508,-0.000078,-0.000756,-0.001174,-0.000795,0.000273,-0.000167,-0.005461,-0.006825,-0.001440,-0.000488,0.004317,-0.000390,0.000180,-0.000081,0.003324,0.002410,-0.002014,-0.000984,0.003376,-0.004502,-0.004492,-0.000759,0.002088,0.001835,0.003256,-0.003652,No
3,-0.002589,0.000161,-0.002651,0.000822,-0.000557,0.000430,-0.001068,0.000267,-0.001065,-0.000384,-0.003146,-0.004059,-0.000888,-0.000551,0.001951,0.000332,-0.000074,0.000395,0.002421,0.001996,-0.001199,-0.000718,0.001951,-0.002525,-0.002177,-0.001032,0.000046,0.001198,0.001590,-0.002395,Yes
4,-0.001835,-0.000902,-0.002752,0.001198,-0.000087,-0.000434,-0.000565,-0.000509,-0.000381,0.000035,-0.002766,-0.004998,-0.001059,-0.000581,0.002341,0.000136,0.000378,0.000424,0.001805,0.002015,-0.001247,-0.000703,0.002030,-0.002067,-0.002401,-0.001062,0.000575,0.001424,0.002104,-0.001594,Yes


In [12]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Contaminated,category,0.0,0.0,2.0,"No, Yes"
1,s12,float64,0.0,0.0,1492.0,"-0.0025, -0.0017, -0.0024, -0.0021, -0.0016, -0.002, -0.0019, -0.0022, -0.0023, -0.0025"
2,s13,float64,0.0,0.0,1409.0,"-0.0006, -0.0007, -0.0007, -0.0006, -0.0007, -0.0006, -0.0007, -0.0006, -0.0006, -0.0006"
3,s14,float64,0.0,0.0,1727.0,"-0.0028, -0.0027, -0.0029, -0.0013, -0.0012, -0.0009, -0.0012, -0.0027, -0.0013, -0.0029"
4,s15,float64,0.0,0.0,1734.0,"-0.002, 0.0009, 0.0009, -0.0019, 0.0008, 0.0006, 0.0006, 0.0008, 0.0009, -0.0016"
5,s16,float64,0.0,0.0,1627.0,"-0.0003, 0.0003, -0.0006, 0.0004, 0.0, 0.0, -0.0002, 0.0001, -0.0003, -0.0008"
6,s21,float64,0.0,0.0,1757.0,"0.0013, 0.0011, -0.0002, 0.0021, 0.0014, 0.0023, 0.0007, 0.0004, 0.0008, 0.0009"
7,s23,float64,0.0,0.0,1661.0,"-0.0008, -0.001, -0.0011, -0.0011, -0.001, -0.001, -0.0012, 0.0011, -0.0008, -0.0011"
8,s24,float64,0.0,0.0,1466.0,"-0.0025, -0.0024, -0.0024, -0.0023, -0.0004, -0.0024, -0.0026, -0.0025, -0.0024, -0.0024"
9,s25,float64,0.0,0.0,1455.0,"-0.0001, -0.0002, 0.0, -0.0005, -0.0003, -0.001, -0.0001, -0.0004, -0.0011, -0.001"


In [13]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
s12,2400.0,-0.002510,0.000724,-0.004887,-0.000780
s13,2400.0,-0.000349,0.000622,-0.003275,0.002130
s14,2400.0,-0.002504,0.001425,-0.006190,0.000550
s15,2400.0,-0.000277,0.001610,-0.004440,0.003166
s16,2400.0,-0.000033,0.000812,-0.002052,0.002970
s21,2400.0,0.000962,0.001012,-0.001671,0.003137
s23,2400.0,-0.000167,0.001154,-0.002553,0.002007
s24,2400.0,-0.001150,0.001177,-0.003422,0.001240
s25,2400.0,-0.000727,0.000629,-0.003324,0.001056
s26,2400.0,-0.000161,0.000329,-0.001513,0.001033


In [14]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column       rank                   
Contaminated 1       No   1200  50.0
             2      Yes   1200  50.0

In [15]:
# Target Distribution
target_df

,count,pct
Contaminated,,
No,1200,50.0
Yes,1200,50.0


## Task Curation

In [16]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [17]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [18]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to hazelnut_spread_contaminant_detection/019d5a5e-88ba-798d-b9ce-5d608eb75376
019d5a5e-88ba-798d-b9ce-5d608eb75376
e689d9a3232f54e29da4bb6c4c515cea599fd03c812efd560ec84b52d7961d60
